# CU28 mixed_context - Modeling Dataset EDA

Notebook narrativo de auditoria para el scope `mixed_context`.


## Objetivo

Realizar el EDA central del dataset final usado para entrenamiento y evaluacion, distinguiendo target upstream, trigger label y senales operativas.


## Alcance

Este analisis describe la ruta oficial reproducible `mixed_context`. Las senales externas se tratan como contexto/proxy. Las variables internas de planta siguen siendo sinteticas salvo carga posterior de cliente.


## Inputs

            - `data/processed/baseline/feature_engineering_modeling__mixed_context.csv`
- `data/processed/baseline/modeling_metadata__mixed_context.json`


## Outputs esperados

            - `reports/tables/eda/modeling_dataset_summary__mixed_context.csv`
- `reports/tables/eda/modeling_dataset_quality__mixed_context.csv`
- `reports/tables/eda/target_by_profile__mixed_context.csv`
- `reports/tables/eda/trigger_balance__mixed_context.csv`
- `reports/figures/eda/target_distribution__mixed_context.png`
- `reports/figures/eda/target_by_profile__mixed_context.png`
- `reports/figures/eda/trigger_balance__mixed_context.png`
- `reports/figures/eda/trigger_by_profile__mixed_context.png`
- `reports/figures/eda/modeling_dataset_timeseries__mixed_context.png`
- `reports/figures/eda/modeling_dataset_correlation__mixed_context.png`


## Limitaciones

Este notebook documenta evidencia reproducible del pipeline oficial, pero no sustituye la revision de codigo, la auditoria de datos de origen ni una certificacion operacional de planta.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.reproducibility.notebook_support import (
    detect_temporal_columns,
    ensure_eda_dirs,
    execution_metadata,
    first_valid_temporal_range,
    load_source_manifests,
    load_tabular_file,
    parse_markdown_table,
    print_frame,
    print_series,
    project_root,
    read_json,
    relative_to_root,
    save_figure,
    save_table,
    sha256_file,
)


In [ ]:
NOTEBOOK_NAME = "05_modeling_dataset_eda.ipynb"
PROJECT_ROOT = project_root()
SCOPE = globals().get("scope", "mixed_context")
REPORT_DIRS = ensure_eda_dirs()
META = execution_metadata(SCOPE)
FIGURES = []
TABLES = []
print(json.dumps(META, indent=2))


## Carga de datos

Las siguientes celdas cargan los artefactos de entrada y muestran verificaciones intermedias antes de producir tablas y graficas.


In [ ]:
modeling_path = PROJECT_ROOT / "data/processed/baseline/feature_engineering_modeling__mixed_context.csv"
metadata_path = PROJECT_ROOT / "data/processed/baseline/modeling_metadata__mixed_context.json"
modeling_df = pd.read_csv(modeling_path)
modeling_df["date"] = pd.to_datetime(modeling_df["date"], errors="coerce")
modeling_meta = read_json(metadata_path)
print(modeling_df.shape)
print_frame("Modeling dataset preview", modeling_df.head(10))


In [ ]:
dataset_summary = pd.DataFrame(
    [
        {
            "rows": len(modeling_df),
            "columns": len(modeling_df.columns),
            "date_min": str(modeling_df["date"].min().date()),
            "date_max": str(modeling_df["date"].max().date()),
            "granularity": "weekly",
            "destination_profiles": int(modeling_df["destination_profile"].nunique()),
        }
    ]
)
column_summary = pd.DataFrame({"column": modeling_df.columns})
print_frame("Dataset summary", dataset_summary)
print_frame("Column summary", column_summary, rows=30)


## Calidad de datos

Se revisan nulos, duplicados, columnas constantes y outliers basicos antes de interpretar el target upstream y la etiqueta trigger.


In [ ]:
null_summary = modeling_df.isna().mean().reset_index()
null_summary.columns = ["column", "missing_pct"]
duplicate_count = int(modeling_df.duplicated().sum())
constant_columns = [column for column in modeling_df.columns if modeling_df[column].nunique(dropna=False) <= 1]
quality_summary = pd.DataFrame(
    [
        {"metric": "duplicate_rows", "value": duplicate_count},
        {"metric": "constant_columns", "value": len(constant_columns)},
    ]
)
print_frame("Null summary", null_summary.sort_values("missing_pct", ascending=False).head(20))
print_frame("Quality summary", quality_summary)


In [ ]:
outlier_rows = []
for column in ["synthetic_procurement_need", "current_inventory_tons", "expected_requirement_tons"]:
    q1 = modeling_df[column].quantile(0.25)
    q3 = modeling_df[column].quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    lower = q1 - 1.5 * iqr
    outlier_rows.append(
        {
            "column": column,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_rows": int(((modeling_df[column] < lower) | (modeling_df[column] > upper)).sum()),
        }
    )
outlier_df = pd.DataFrame(outlier_rows)
print_frame("Basic outlier audit", outlier_df)
display(outlier_df)


In [ ]:
target_distribution = modeling_df["synthetic_procurement_need"].describe().reset_index()
target_distribution.columns = ["metric", "value"]
target_by_profile = modeling_df.groupby("destination_profile")["synthetic_procurement_need"].agg(["count", "mean", "median", "min", "max"]).reset_index()
print_frame("Target distribution", target_distribution, rows=20)
print_frame("Target by profile", target_by_profile, rows=20)


In [ ]:
trigger_balance = modeling_df["purchase_trigger_label"].value_counts(normalize=True).reset_index()
trigger_balance.columns = ["purchase_trigger_label", "share"]
trigger_by_profile = modeling_df.groupby("destination_profile")["purchase_trigger_label"].mean().reset_index()
print_frame("Trigger balance", trigger_balance, rows=20)
print_frame("Trigger by profile", trigger_by_profile, rows=20)


In [ ]:
trigger_vs_inventory = modeling_df.groupby(pd.cut(modeling_df["current_inventory_tons"], bins=10))["purchase_trigger_label"].mean().reset_index()
trigger_vs_requirement = modeling_df.groupby(pd.cut(modeling_df["expected_requirement_tons"], bins=10))["purchase_trigger_label"].mean().reset_index()
print_frame("Trigger vs inventory bins", trigger_vs_inventory, rows=20)
print_frame("Trigger vs requirement bins", trigger_vs_requirement, rows=20)


## Graficas

Las graficas siguientes muestran comportamiento del target upstream, balance trigger y senales temporales semanales.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(modeling_df["synthetic_procurement_need"], bins=25, color="#355c7d", edgecolor="white")
ax.set_title("synthetic_procurement_need distribution")
ax.set_xlabel("tons")
FIGURES.append(save_figure(fig, "target_distribution__mixed_context.png"))
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(target_by_profile["destination_profile"], target_by_profile["mean"], color="#6c5b7b")
ax.set_title("synthetic_procurement_need by destination profile")
ax.set_ylabel("mean tons")
FIGURES.append(save_figure(fig, "target_by_profile__mixed_context.png"))
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(trigger_balance["purchase_trigger_label"].astype(str), trigger_balance["share"], color="#c06c84")
ax.set_title("Trigger label balance")
ax.set_ylabel("share")
FIGURES.append(save_figure(fig, "trigger_balance__mixed_context.png"))
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(trigger_by_profile["destination_profile"], trigger_by_profile["purchase_trigger_label"], color="#f67280")
ax.set_title("Trigger rate by destination profile")
ax.set_ylabel("trigger_rate")
FIGURES.append(save_figure(fig, "trigger_by_profile__mixed_context.png"))
plt.close(fig)


In [ ]:
weekly_signals = modeling_df.groupby("date")[["synthetic_procurement_need", "current_inventory_tons", "expected_requirement_tons"]].mean().reset_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(weekly_signals["date"], weekly_signals["synthetic_procurement_need"], label="synthetic_procurement_need")
ax.plot(weekly_signals["date"], weekly_signals["expected_requirement_tons"], label="expected_requirement_tons")
ax.plot(weekly_signals["date"], weekly_signals["current_inventory_tons"], label="current_inventory_tons")
ax.set_title("Weekly modeling signals")
ax.legend()
FIGURES.append(save_figure(fig, "modeling_dataset_timeseries__mixed_context.png"))
plt.close(fig)


In [ ]:
corr_columns = [
    "synthetic_procurement_need",
    "purchase_trigger_label",
    "current_inventory_tons",
    "expected_requirement_tons",
    "lead_time_days",
    "safety_coverage_days",
]
corr = modeling_df[corr_columns].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Reduced correlation matrix")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
FIGURES.append(save_figure(fig, "modeling_dataset_correlation__mixed_context.png"))
plt.close(fig)


## Interpretacion

La granularidad semanal permite combinar proxies externos con una capa sintetica de planta sin pretender que el dataset sea un historico observado de compras. `synthetic_procurement_need` sigue siendo el target upstream y no la cantidad final recomendada.


In [ ]:
TABLES.append(save_table(dataset_summary, "modeling_dataset_summary__mixed_context.csv"))
TABLES.append(save_table(outlier_df, "modeling_dataset_quality__mixed_context.csv"))
TABLES.append(save_table(target_by_profile, "target_by_profile__mixed_context.csv"))
TABLES.append(save_table(trigger_balance, "trigger_balance__mixed_context.csv"))
RESULT = {
    "notebook": NOTEBOOK_NAME,
    "tables": TABLES,
    "figures": FIGURES,
    "findings": [
        "The modeling dataset is weekly and combines proxy, synthetic and derived variables.",
        "synthetic_procurement_need varies materially by destination profile and supports upstream modeling.",
        "purchase_trigger_label tracks operational stress through inventory, requirement and lead-time relationships.",
    ],
    "limitations": [
        "The dataset remains mixed and derived; it should not be read as an observed purchase ledger.",
    ],
}
print(json.dumps(RESULT, indent=2))


## Limitaciones

La calidad del dataset final depende de decisiones de simulacion y armonizacion semanal. El objetivo del notebook es hacer visibles esas decisiones y sus efectos estadisticos basicos.


## Concluson final

Este es el EDA central del dataset modelable: resume cobertura temporal, calidad, dispersion por perfil y comportamiento conjunto de target upstream y trigger.
